# Laughter Detection on AWS Inferentia2 with Neuron SDK

This notebook traces, benchmarks, and validates the [LaughterSegmentation](https://github.com/omine-me/LaughterSegmentation) model on AWS Inferentia2 using the Neuron SDK. The model is a Wav2Vec2-based audio frame classifier (~315M params, FP32) that detects laughter segments in speech audio.

**Instance**: inf2.xlarge (1 Inferentia2 chip, 2 NeuronCores)
**Time to complete**: ~15 minutes
**Prerequisites**: Deep Learning AMI Neuron (Ubuntu 24.04)

### Quick Start

```bash
# Activate pre-installed Neuron venv
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/bin/activate
pip install jupyter
jupyter notebook --no-browser --port 8888
```

### trn2 Usage

This notebook is configured for inf2. To run on trn2.3xlarge, uncomment the `--lnc` compiler arg in Step 2. See comments in the compilation cell.

## Model & Instance Specifications

| Property | Value |
|----------|-------|
| Model | omine-me/LaughterSegmentation |
| Architecture | Wav2Vec2ForAudioFrameClassification |
| Parameters | ~315M (FP32) |
| Base model | jonatasgrosman/wav2vec2-large-xlsr-53-english |
| Input | 7-second audio windows at 16 kHz (112,000 samples) |
| Output | 349 per-frame binary predictions (laughter / not-laughter) |

| Software | Version |
|----------|--------|
| DLAMI | Deep Learning AMI Neuron (Ubuntu 24.04) 20260227 |
| Neuron SDK | 2.28 |
| PyTorch | 2.9 |
| Transformers | 4.x |

> **Important**: Wav2Vec2 uses `weight_norm` parametrizations on its positional convolution embedding. These must be removed before `torch_neuronx.trace()` or the compiler will crash with a PjRt buffer null error. This notebook handles this automatically.

---
## Step 0: Setup & Environment Check

Install required packages and verify the Neuron environment.

In [1]:
import subprocess, sys

# Install dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "safetensors", "librosa", "scipy", "pydub"])

import torch
import torch_neuronx

print(f"PyTorch:      {torch.__version__}")
print(f"torch-neuronx: {torch_neuronx.__version__}")
print()
subprocess.run(["neuron-ls"])

PyTorch:      2.9.0+cu128
torch-neuronx: 2.9.0.2.12.22436+0f1dac25



instance-type: inf2.xlarge
instance-id: i-0f0fd1a81f421fbc9
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 2      | 0-1      | 32 GB  | 0000:00:1f.0 | 0-3      | -1   |
+--------+--------+----------+--------+--------------+----------+------+


CompletedProcess(args=['neuron-ls'], returncode=0)

---
## Step 1: Download Model Weights

Download the fine-tuned LaughterSegmentation weights (1.26 GB safetensors) from HuggingFace, then load the model with the correct architecture configuration.

**Key detail**: The safetensors file stores weights with an `audio_model.` prefix (from the training wrapper class). We strip this prefix when loading into the bare HuggingFace `Wav2Vec2ForAudioFrameClassification`.

In [2]:
import os
import torch.nn.utils.parametrize as parametrize
import safetensors.torch
from transformers import Wav2Vec2Config, Wav2Vec2ForAudioFrameClassification

MODEL_DIR = "./models"
WEIGHTS_PATH = os.path.join(MODEL_DIR, "model.safetensors")
AUDIO_MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-english"

# Download weights if not present
if not os.path.exists(WEIGHTS_PATH):
    os.makedirs(MODEL_DIR, exist_ok=True)
    subprocess.check_call(["hf", "download", "omine-me/LaughterSegmentation",
                           "model.safetensors", "--local-dir", MODEL_DIR])

# Load model from config (avoids downloading full base model checkpoint)
config = Wav2Vec2Config.from_pretrained(AUDIO_MODEL_NAME)
config.num_labels = 1
config.problem_type = "single_label_classification"
model = Wav2Vec2ForAudioFrameClassification(config)

# Load fine-tuned weights, stripping the "audio_model." prefix
state_dict = safetensors.torch.load_file(WEIGHTS_PATH, device="cpu")
prefix = "audio_model."
stripped = {(k[len(prefix):] if k.startswith(prefix) else k): v
            for k, v in state_dict.items()}
model.load_state_dict(stripped)
model.eval()

# CRITICAL: Remove weight_norm parametrizations before tracing.
# Wav2Vec2 uses weight_norm on pos_conv_embed.conv, which crashes
# torch_neuronx.trace() on SDK 2.28 with a PjRt buffer null error.
for name, module in model.named_modules():
    if hasattr(module, "parametrizations"):
        for param_name in list(module.parametrizations.keys()):
            print(f"Removing parametrization: {name}.{param_name}")
            parametrize.remove_parametrizations(module, param_name)

param_count = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded: {param_count / 1e6:.1f}M parameters")

Download complete. Moving file to models/model.safetensors


models/model.safetensors


config.json: 0.00B [00:00, ?B/s]

Removing parametrization: wav2vec2.encoder.pos_conv_embed.conv.weight

Model loaded: 315.4M parameters


---
## Step 2: Compile for Neuron

Trace the model with `torch_neuronx.trace()` across multiple batch sizes. We use:

- `--model-type transformer` -- optimizes attention patterns
- `--auto-cast matmult` -- casts FP32 matmuls to BF16 for ~2x throughput with negligible accuracy loss
- `inline_weights_to_neff=True` -- embeds weights in the compiled artifact for faster loading

Each batch size requires a separate compilation (~2 minutes each).

In [ ]:
import time

INPUT_SEC = 7
SAMPLE_RATE = 16000
INPUT_SAMPLES = INPUT_SEC * SAMPLE_RATE  # 112,000

BATCH_SIZES = [1, 2, 4, 8]
COMPILED_DIR = "./compiled"
os.makedirs(COMPILED_DIR, exist_ok=True)

compiler_args = [
    "--model-type", "transformer",
    "--optlevel", "2",
    "--auto-cast", "matmult",
    # --- trn2 only: uncomment one of the following ---
    # "--lnc", "1",  # LNC=1: model on 1 NeuronCore
    # "--lnc", "2",  # LNC=2: model spans 2 NeuronCores, better per-model latency
]

compiled_models = {}

for bs in BATCH_SIZES:
    save_path = os.path.join(COMPILED_DIR, f"laughter_bs{bs}.pt")

    if os.path.exists(save_path):
        print(f"BS={bs}: Loading cached {save_path}")
        compiled_models[bs] = torch.jit.load(save_path)
        continue

    print(f"BS={bs}: Compiling...", end=" ", flush=True)
    example_input = torch.randn(bs, INPUT_SAMPLES)

    t0 = time.time()
    model_neuron = torch_neuronx.trace(
        model, example_input,
        compiler_args=compiler_args,
        inline_weights_to_neff=True,
    )
    compile_time = time.time() - t0

    torch.jit.save(model_neuron, save_path)
    file_mb = os.path.getsize(save_path) / (1024 * 1024)
    print(f"{compile_time:.0f}s, {file_mb:.0f} MB")
    compiled_models[bs] = model_neuron

print(f"\nCompiled {len(compiled_models)} models.")

BS=1: Compiling... 

.

.

.

.

...

.

.

.


Compiler status PASS


374s, 506 MB

Compiled 4 models.


---
## Step 3: Benchmark

Measure latency and throughput for each batch size. We report:

- **Latency**: mean, p50, p95, p99 in milliseconds
- **Throughput**: audio windows processed per second
- **Real-time factor (RTF)**: seconds of audio processed per second of wall time (>1 = faster than real-time)

In [4]:
import numpy as np

WARMUP = 10
ITERATIONS = 100

results = []

for bs in BATCH_SIZES:
    model_n = compiled_models[bs]
    x = torch.randn(bs, INPUT_SAMPLES)

    # Warmup
    for _ in range(WARMUP):
        model_n(x)

    # Benchmark
    latencies = []
    for _ in range(ITERATIONS):
        t0 = time.time()
        model_n(x)
        latencies.append((time.time() - t0) * 1000)

    lat = np.array(latencies)
    throughput = bs / (lat.mean() / 1000)
    rtf = throughput * INPUT_SEC

    results.append({
        "bs": bs,
        "mean": lat.mean(), "p50": np.percentile(lat, 50),
        "p95": np.percentile(lat, 95), "p99": np.percentile(lat, 99),
        "throughput": throughput, "rtf": rtf,
    })

# Print results table
print(f"{'BS':>4} | {'Mean(ms)':>9} | {'p50(ms)':>8} | {'p95(ms)':>8} | {'p99(ms)':>8} | {'W/s':>8} | {'RTF':>8}")
print("-" * 72)
for r in results:
    print(f"{r['bs']:>4} | {r['mean']:>9.2f} | {r['p50']:>8.2f} | {r['p95']:>8.2f} | "
          f"{r['p99']:>8.2f} | {r['throughput']:>8.1f} | {r['rtf']:>7.0f}x")

  BS |  Mean(ms) |  p50(ms) |  p95(ms) |  p99(ms) |      W/s |      RTF
------------------------------------------------------------------------
   1 |     18.44 |    17.75 |    23.13 |    24.11 |     54.2 |     380x
   2 |     21.27 |    21.28 |    21.47 |    21.50 |     94.0 |     658x
   4 |     42.05 |    42.04 |    42.11 |    42.15 |     95.1 |     666x
   8 |     83.93 |    83.91 |    84.13 |    84.19 |     95.3 |     667x


---
## Step 3b: DataParallel Benchmark (Full Instance)

`torch_neuronx.DataParallel` loads the same compiled model onto each available NeuronCore and runs
them in parallel. Each core processes its own full batch independently, multiplying total throughput
by the number of cores.

On inf2.xlarge (2 NeuronCores), this gives each core its own batch, doubling throughput.

**Important**: `DataParallel` defaults to `num_workers=2` threads. This works for 2-core instances,
but on instances with more cores, set `model_dp.num_workers = num_cores` for full parallelism.

In [5]:
import json as _json

def get_neuron_core_count():
    """Detect the number of Neuron cores using neuron-ls."""
    try:
        result = subprocess.run(
            ["neuron-ls", "--json-output"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            devices = _json.loads(result.stdout)
            return sum(d["nc_count"] for d in devices)
    except Exception:
        pass
    return 1

num_cores = get_neuron_core_count()
print(f"Neuron cores detected: {num_cores}")

# Use the best single-core batch size for DataParallel
DP_BATCH_SIZE = 2

if num_cores > 1:
    # Load the compiled model onto all cores
    model_dp = torch_neuronx.DataParallel(compiled_models[DP_BATCH_SIZE])
    model_dp.num_workers = num_cores

    # Each core gets DP_BATCH_SIZE, so total input is DP_BATCH_SIZE * num_cores
    dp_total_batch = DP_BATCH_SIZE * num_cores
    dp_input = torch.randn(dp_total_batch, INPUT_SAMPLES)
    print(f"DataParallel input shape: {dp_input.shape}  ({DP_BATCH_SIZE} per core x {num_cores} cores)")

    print(f"\nWarming up DataParallel...")
    for _ in range(WARMUP):
        model_dp(dp_input)

    print(f"Benchmarking DataParallel ({ITERATIONS} iterations)...")
    dp_latencies = []
    for _ in range(ITERATIONS):
        t0 = time.time()
        model_dp(dp_input)
        dp_latencies.append((time.time() - t0) * 1000)

    dp_lat = np.array(dp_latencies)
    dp_throughput = dp_total_batch / (dp_lat.mean() / 1000)
    dp_rtf = dp_throughput * INPUT_SEC

    # Find single-core results for comparison
    single_result = [r for r in results if r['bs'] == DP_BATCH_SIZE][0]

    print(f"\n=== DataParallel Results ({num_cores} cores) ===")
    print(f"Latency  mean: {dp_lat.mean():.2f} ms  p95: {np.percentile(dp_lat, 95):.2f} ms")
    print(f"Throughput:    {dp_throughput:.1f} windows/sec  ({dp_rtf:.0f}x real-time)")
    print(f"\n=== Comparison ===")
    print(f"Single core (BS={DP_BATCH_SIZE}):  {single_result['throughput']:.1f} W/s  ({single_result['rtf']:.0f}x RT)")
    print(f"DataParallel ({num_cores} cores):  {dp_throughput:.1f} W/s  ({dp_rtf:.0f}x RT)")
    print(f"Speedup: {dp_throughput / single_result['throughput']:.2f}x")
else:
    print("Only 1 Neuron core available - skipping DataParallel benchmark.")
    print("Use an instance with multiple cores (e.g., inf2.xlarge has 2) for DataParallel.")

Neuron cores detected: 2


DataParallel input shape: torch.Size([4, 112000])  (2 per core x 2 cores)

Warming up DataParallel...


Benchmarking DataParallel (100 iterations)...



=== DataParallel Results (2 cores) ===
Latency  mean: 22.84 ms  p95: 22.94 ms
Throughput:    175.2 windows/sec  (1226x real-time)

=== Comparison ===
Single core (BS=2):  94.0 W/s  (658x RT)
DataParallel (2 cores):  175.2 W/s  (1226x RT)
Speedup: 1.86x


---
## Step 4: Accuracy Validation

Compare the Neuron-compiled model output against a CPU reference to verify that `--auto-cast matmult` does not degrade laughter detection accuracy.

We generate random audio inputs of varying characteristics and measure:
- **Cosine similarity** between raw logit vectors
- **Frame-level prediction agreement** (after sigmoid + 0.5 threshold)

In [6]:
neuron_model = compiled_models[1]  # BS=1 for validation

# Generate diverse test inputs
torch.manual_seed(42)
test_inputs = {
    "random_normal":  torch.randn(1, INPUT_SAMPLES),
    "quiet_noise":    torch.randn(1, INPUT_SAMPLES) * 0.01,
    "loud_signal":    torch.randn(1, INPUT_SAMPLES) * 5.0,
    "sine_440hz":     torch.sin(2 * torch.pi * 440 * torch.linspace(0, INPUT_SEC, INPUT_SAMPLES)).unsqueeze(0),
    "silence":        torch.zeros(1, INPUT_SAMPLES),
}

print(f"{'Input':>16} | {'Cosine Sim':>10} | {'Max Error':>10} | {'Mean Error':>10} | {'Agreement':>10} | {'CPU Laugh':>10} | {'NRN Laugh':>10}")
print("-" * 98)

for name, x in test_inputs.items():
    # CPU reference
    with torch.no_grad():
        cpu_out = model(input_values=x)
    cpu_logits = cpu_out.logits.squeeze(-1)  # [1, 349]

    # Neuron
    neuron_out = neuron_model(x)
    if isinstance(neuron_out, dict):
        neuron_logits = neuron_out["logits"].squeeze(-1)
    elif isinstance(neuron_out, (tuple, list)):
        neuron_logits = neuron_out[0].squeeze(-1)
    else:
        neuron_logits = neuron_out.squeeze(-1)

    cpu_flat = cpu_logits.flatten().float()
    nrn_flat = neuron_logits.flatten().float()

    cosine = torch.nn.functional.cosine_similarity(cpu_flat.unsqueeze(0), nrn_flat.unsqueeze(0)).item()
    max_err = (cpu_flat - nrn_flat).abs().max().item()
    mean_err = (cpu_flat - nrn_flat).abs().mean().item()

    cpu_preds = (torch.sigmoid(cpu_logits) >= 0.5).int()
    nrn_preds = (torch.sigmoid(neuron_logits) >= 0.5).int()
    agree = (cpu_preds == nrn_preds).float().mean().item()

    print(f"{name:>16} | {cosine:>10.6f} | {max_err:>10.4f} | {mean_err:>10.4f} | "
          f"{agree * 100:>9.2f}% | {cpu_preds.sum().item():>10} | {nrn_preds.sum().item():>10}")

           Input | Cosine Sim |  Max Error | Mean Error |  Agreement |  CPU Laugh |  NRN Laugh
--------------------------------------------------------------------------------------------------


   random_normal |   1.000000 |     0.0195 |     0.0017 |    100.00% |          0 |          0


     quiet_noise |   1.000000 |     0.0093 |     0.0010 |    100.00% |          0 |          0


     loud_signal |   1.000000 |     0.0100 |     0.0047 |    100.00% |          0 |          0


      sine_440hz |   1.000000 |     0.0178 |     0.0030 |    100.00% |          0 |          0


         silence |   0.999999 |     0.0288 |     0.0030 |    100.00% |          0 |          0


---
## Step 5: End-to-End Inference Demo

Run the full laughter detection pipeline on a sample audio file:
1. Load audio at 16 kHz
2. Slice into 7-second windows with 2-second overlap
3. Batch inference through the Neuron model
4. Sigmoid + threshold + merge overlapping detections
5. Output JSON with laughter timestamps

We use a librosa built-in audio sample for the demo.

In [7]:
import json
import librosa

# Load a sample audio file (librosa built-in)
audio_path = librosa.example("trumpet")
audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
duration = len(audio) / SAMPLE_RATE
print(f"Audio: {duration:.1f}s, {len(audio)} samples")

# Windowed inference
OVERLAP_SEC = 2.0
stride = int(SAMPLE_RATE * (INPUT_SEC - OVERLAP_SEC))
neuron_bs2 = compiled_models[2]  # Use BS=2 for throughput
batch_size = 2

all_preds = []
laughter_events = {}
event_idx = 0
t0 = time.time()

with torch.no_grad():
    for start in range(0, len(audio), stride * batch_size):
        windows = []
        actual = 0
        for i in range(batch_size):
            w_start = start + i * stride
            w = audio[w_start:w_start + INPUT_SAMPLES]
            if len(w) == 0:
                break
            if len(w) < INPUT_SAMPLES:
                w = np.append(w, np.zeros(INPUT_SAMPLES - len(w)))
            windows.append(w)
            actual += 1

        if actual == 0:
            break

        # Pad to compiled batch size
        while len(windows) < batch_size:
            windows.append(np.zeros(INPUT_SAMPLES))

        x = torch.from_numpy(np.array(windows)).float()
        out = neuron_bs2(x)
        logits = (out["logits"] if isinstance(out, dict) else out).squeeze(-1)
        preds = (torch.sigmoid(logits[:actual].float()) >= 0.5).int()

        for bi in range(actual):
            w_start_sec = (start + bi * stride) / SAMPLE_RATE
            frame_pred = preds[bi].cpu().numpy()
            frame_count = len(frame_pred)
            in_laugh = False
            seg_start = None

            for fi, f in enumerate(frame_pred):
                if f == 1 and not in_laugh:
                    seg_start = fi
                    in_laugh = True
                elif (f == 0 or fi == frame_count - 1) and in_laugh:
                    seg_end = fi if f == 0 else fi + 1
                    laughter_events[str(event_idx)] = {
                        "start_sec": round(w_start_sec + (INPUT_SEC / frame_count) * seg_start, 3),
                        "end_sec": round(w_start_sec + (INPUT_SEC / frame_count) * seg_end, 3),
                    }
                    event_idx += 1
                    in_laugh = False

inference_time = time.time() - t0

# Merge overlapping events from overlapping windows
merged = {}
for evt in sorted(laughter_events.values(), key=lambda e: e["start_sec"]):
    if not merged:
        merged["0"] = evt.copy()
    else:
        last = list(merged.values())[-1]
        if evt["start_sec"] <= last["end_sec"]:
            last["end_sec"] = max(last["end_sec"], evt["end_sec"])
        else:
            merged[str(len(merged))] = evt.copy()

# Remove short events (<0.2s)
merged = {k: v for k, v in merged.items() if v["end_sec"] - v["start_sec"] >= 0.2}
merged = {str(i): v for i, v in enumerate(merged.values())}

print(f"\nDetected {len(merged)} laughter segment(s) in {inference_time:.3f}s ({duration / inference_time:.0f}x real-time):")
for idx, seg in merged.items():
    print(f"  [{idx}] {seg['start_sec']:.2f}s - {seg['end_sec']:.2f}s ({seg['end_sec'] - seg['start_sec']:.2f}s)")

if not merged:
    print("  (no laughter detected)")

# Save results
os.makedirs("./output", exist_ok=True)
with open("./output/laughter_results.json", "w") as f:
    json.dump(merged, f, indent=2)
print(f"\nSaved to ./output/laughter_results.json")

Audio: 5.3s, 85335 samples

Detected 1 laughter segment(s) in 0.025s (217x real-time):
  [0] 0.28s - 1.58s (1.30s)

Saved to ./output/laughter_results.json


---
## Summary

This notebook demonstrated:

1. **Tracing** Wav2Vec2ForAudioFrameClassification on Neuron with `torch_neuronx.trace()`
2. **Single-core benchmarking** across batch sizes -- throughput saturates around BS=2-4 on inf2
3. **DataParallel benchmarking** -- `torch_neuronx.DataParallel` for full-instance throughput across all NeuronCores
4. **Accuracy validation** -- cosine similarity ~1.0, prediction agreement 100% with `--auto-cast matmult`
5. **End-to-end inference** -- audio file to laughter timestamps at 100x+ real-time

### Key Findings

- **weight_norm fix required**: `parametrize.remove_parametrizations()` must be called before tracing any model with weight_norm on SDK 2.28+
- **`--auto-cast matmult` is safe**: negligible accuracy impact for this binary classification task
- **`inline_weights_to_neff=True`** recommended for single-model deployment

### trn2 Notes

To run on trn2.3xlarge, add `"--lnc", "1"` or `"--lnc", "2"` to `compiler_args` in Step 2:
- **LNC=1**: Model uses 1 NeuronCore. May be more efficient for smaller models.
- **LNC=2**: Model spans 2 NeuronCores. Better per-model latency, may not give maximum throughput.